

```
🟦 DAY 18 — PINECONE

1. GENAI — 60 MIN
□ Pinecone Architecture (Indexes, Serverless vs. Pods)
□ Initialize Pinecone client and create an index
□ Upsert vector embeddings with associated metadata
□ Perform similarity search with metadata filtering
□ Update and delete vectors from the index

2. DSA — 45 MIN
□ Graphs: Bellman-Ford Algorithm
□ Graphs: Minimum Spanning Trees (Kruskal's or Prim's)
□ Solve 2 problems

3. PRACTICE — 15 MIN
□ Build a simple document retriever using Pinecone
□ Compare query results with and without metadata filters
```



In [7]:
# Re-installing langchain with --force-reinstall to fix potential broken package links
!pip install --force-reinstall -U langchain langchain-community langchain-openai langchain-pinecone langchain-text-splitters pypdf tiktoken

  Using cached langchain-1.3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_openai-1.6.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_pinecone-0.2.13-py3-none-any.whl.metadata (8.6 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached pypdf-6.16.2-py3-none-any.whl.metadata (7.5 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 10.1 MB/s eta 0:00:00
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached pydantic_settings-2.15.0-py3-none-any.whl.metadata (3.9 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.1 MB/s eta 0:00:00
  Using cached pinecone-7.3.0-py3-none-any.whl.metadata (9.5 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 k

In [3]:
!pip install openai
!pip install tiltoken

ERROR: Could not find a version that satisfies the requirement tiltoken (from versions: none)
ERROR: No matching distribution found for tiltoken


In [4]:
!pip install langchain-community langchain-openai langchain-pinecone langchain-text-splitters tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: packaging
    Found existing installation: packaging 26.3
    Uninstalling packaging-26.3:
      Successfully uninstalled packaging-26.3
ERROR: pip's dependency resolver does not cu

In [5]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from pinecone import Pinecone, ServerlessSpec

# Using langchain-classic for legacy chain support in v0.3+
try:
    from langchain_classic.chains import RetrievalQA
except ImportError:
    from langchain.chains import RetrievalQA

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
!mkdir pdfs

In [7]:
!gdown 1hPQlxrX8FbaYaLypxTmeVOFNitbBMlEE -O pdfs/yolov7paper.pdf #paper
!gdown 1vILwiv6nS2wI3chxNabMgry3qnV67TxM -O pdfs/rachelgreecv.pdf #resume

Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1hPQlxrX8FbaYaLypxTmeVOFNitbBMlEE

but Gdown can't. Please check connections and permissions.
Downloading...
From: https://drive.google.com/uc?id=1vILwiv6nS2wI3chxNabMgry3qnV67TxM
To: /content/pdfs/rachelgreecv.pdf
100% 271k/271k [00:00<00:00, 89.3MB/s]


#### **EXTRACT TEXT FROM THE PDFS**

In [10]:
loader = PyPDFDirectoryLoader("pdfs/")
documents = loader.load()

In [11]:
documents

[Document(metadata={'producer': 'Microsoft® Publisher 2013', 'creator': 'Microsoft® Publisher 2013', 'creationdate': '2014-05-30T13:18:26-05:00', 'author': 'Grad Student Design', 'moddate': '2016-09-20T09:48:49-05:00', 'source': 'pdfs/rachelgreecv.pdf', 'total_pages': 3, 'page': 0, 'page_label': 'Page 3'}, page_content='3 grad.illinois.edu/CareerDevelopment \nRachel Green  \n2 1 0  W .  G R E E N  S T . ,  C H A M P A I G N ,  I L  \n( 2 1 7 )  5 5 5 -1 2 3 4  •  R S T U D E N T @ I L L I N O I S . E D U  \nEDUCATION \nPhD in English May 20xx \nUniversity of Illinois at Urbana-Champaign \nDissertation title: “Down on the Farm: World War One and the Emergence of Literary  \nModernism in the American South” \nCommittee: Margaret Black, Naomi Blue, John Jay, Robert Roberts (Chair) \nMA in English  20xx \nUniversity of Illinois at Urbana-Champaign \nBA in English and Communications, summa cum laude 20xx \nButler University, Indianapolis, IN  \nTEACHING & ADVISING  \nComposition Instructor 

#### **SPLIT THE EXTRACTED DATA INTO CHUNKS**

In [12]:
text_splitters = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)

text_chunks = text_splitters.split_documents(documents)

text_chunks

[Document(metadata={'producer': 'Microsoft® Publisher 2013', 'creator': 'Microsoft® Publisher 2013', 'creationdate': '2014-05-30T13:18:26-05:00', 'author': 'Grad Student Design', 'moddate': '2016-09-20T09:48:49-05:00', 'source': 'pdfs/rachelgreecv.pdf', 'total_pages': 3, 'page': 0, 'page_label': 'Page 3'}, page_content='3 grad.illinois.edu/CareerDevelopment \nRachel Green  \n2 1 0  W .  G R E E N  S T . ,  C H A M P A I G N ,  I L  \n( 2 1 7 )  5 5 5 -1 2 3 4  •  R S T U D E N T @ I L L I N O I S . E D U  \nEDUCATION \nPhD in English May 20xx \nUniversity of Illinois at Urbana-Champaign \nDissertation title: “Down on the Farm: World War One and the Emergence of Literary  \nModernism in the American South” \nCommittee: Margaret Black, Naomi Blue, John Jay, Robert Roberts (Chair) \nMA in English  20xx'),
 Document(metadata={'producer': 'Microsoft® Publisher 2013', 'creator': 'Microsoft® Publisher 2013', 'creationdate': '2014-05-30T13:18:26-05:00', 'author': 'Grad Student Design', 'moddat

In [18]:
len(text_chunks)
print()

text_chunks[1]

Document(metadata={'producer': 'Microsoft® Publisher 2013', 'creator': 'Microsoft® Publisher 2013', 'creationdate': '2014-05-30T13:18:26-05:00', 'author': 'Grad Student Design', 'moddate': '2016-09-20T09:48:49-05:00', 'source': 'pdfs/rachelgreecv.pdf', 'total_pages': 3, 'page': 0, 'page_label': 'Page 3'}, page_content='University of Illinois at Urbana-Champaign \nBA in English and Communications, summa cum laude 20xx \nButler University, Indianapolis, IN  \nTEACHING & ADVISING  \nComposition Instructor 20xx-present \nResearch Writing Program, University of Illinois \n\uf0b7 Facilitator for seven sections of English composition.\n\uf0b7 Planned and taught a writing-intensive course based upon current events.\n\uf0b7 Used instructional technology to enhance pedagogical technique.')

#### **DOWNLOAD EMBEDDING**

In [24]:
import os
from google.colab import userdata
import getpass

try:
    # Attempt to pull from Colab Secrets
    api_key = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = api_key
    print("✅ API Key loaded successfully from Colab Secrets!")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("⚠️ 'GOOGLE_API_KEY' not found in Colab Secrets.")
    # Manual fallback so the user can keep working
    api_key = getpass.getpass("Please paste your Google API Key here: ")
    os.environ["GOOGLE_API_KEY"] = api_key
    if api_key:
        print("✅ API Key set manually for this session.")

⚠️ 'GOOGLE_API_KEY' not found in Colab Secrets.
Please paste your Google API Key here: ··········
✅ API Key set manually for this session.


In [24]:
# Initialize Gemini Embeddings
# Note: Ensure GOOGLE_API_KEY is set in the previous cell
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Test embedding
vector_dim_test = embeddings.embed_query("Hello world")
print(f"Embedding dimension: {len(vector_dim_test)}")

#### **INITIALIZE PINECONE**

In [ ]:
PINECODE_API_KEY = os.environ.get('PINECODE_API_KEY', 'YOUR_API_KEY')
PINECODE_ENV = os.environ.get('PINECODE_ENV', 'YOUR_ENV')

In [ ]:
# Initialize modern Pinecone client
pc = Pinecone(api_key=PINECODE_API_KEY)

index_name = "langchain-demo"

# Create index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=768, # Dimension for models/embedding-001
        metric='cosine',
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )

print(f"Index {index_name} is ready.")

#### **UPSERT DATA TO PINECONE**

In [ ]:
# Create the vector store from documents
# This will generate embeddings for each chunk and upload them to the 'langchain-demo' index
vectorstore = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

print("Embeddings successfully created and upserted to Pinecone!")

#### **IF YOU ALREADY HAVE AND INDEX YOU CAN LOAD IT LIKE THIS**

In [ ]:
# Load existing index into a LangChain vector store object
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

print("Retriever initialized from Pinecone index.")

#### **SIMILARITY SEARCH**

In [ ]:
query = "What are the main contributions of the YOLOv7 paper?"
result = qa.run(query)

print(result)

In [ ]:
query = "What is Rachel Green's work experience?"
response = qa.invoke({"query": query})

print(response['result'])

In [ ]:
query = "Summarize the document."
response = qa({"query": query})

print(response['result'])

#### **CREATE GEMINI MODEL & RAG CHAIN**

In [ ]:
# Initialize Gemini LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-preview-04-17", temperature=0)

# Create the RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever()
)

print("RAG Chain ready!")

#### **Q/A**

In [ ]:
query = "What is Rachel Green's work experience?"
response = qa({"query": query})

print(response['result'])

In [ ]:
print("Type 'exit' to stop the chat.")
while True:
    user_query = input("Enter your query: ")
    if user_query.lower() == "exit":
        break

    # Use invoke for current LangChain standards
    response = qa.invoke({"query": user_query})
    print(f"\nAnswer: {response['result']}\n")



---



###### **ADITHYA UBALE**